In [0]:
# ============================================================
# NOTEBOOK 04 — RANDOM FOREST + ÉVALUATION
# Section 4.3 + Section 5 du papier Belcastro et al. (2016)
# ============================================================

# CELLULE 1 — Initialisation
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, \
                                   BinaryClassificationEvaluator
import pandas as pd

spark = SparkSession.builder \
    .appName("FlightDelay_Notebook04") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print(f"✅ Spark {spark.version} prêt !")

# ============================================================
# CELLULE 2 — Features
# Section 4.3 : W_o + W_d = vecteur météo
# 8 variables × 13 slots × 2 aéroports = 208 features
# ============================================================

METEO_VARS = ["Temp", "Hum", "WDir", "WSpd",
              "Pres", "Sky", "Vis", "WType"]
SLOTS = range(13)

# Construire liste features orig_* et dest_*
FEATURES_ORIG = [f"orig_{v}_{i}h"
                 for i in SLOTS for v in METEO_VARS]
FEATURES_DEST = [f"dest_{v}_{i}h"
                 for i in SLOTS for v in METEO_VARS]
ALL_FEATURES  = FEATURES_ORIG + FEATURES_DEST

print(f"📋 Features totales : {len(ALL_FEATURES)}")
print(f"   orig : {len(FEATURES_ORIG)} | dest : {len(FEATURES_DEST)}")

# ============================================================
# CELLULE 3 — Pipeline ML
# Imputer → VectorAssembler → RandomForest
# Section 4.3 : RF avec 100 arbres, maxDepth=10
# ============================================================

def build_pipeline(label_col, features):
    """
    Construit le pipeline ML complet :
    1. Imputer  → remplace NULL par mean
    2. Assembler → vecteur features
    3. RandomForest
    """
    # Garder seulement les features présentes
    imputer = Imputer(
        inputCols=features,
        outputCols=[f"{c}_imp" for c in features],
        strategy="mean"
    )
    features_imp = [f"{c}_imp" for c in features]

    assembler = VectorAssembler(
        inputCols=features_imp,
        outputCol="features",
        handleInvalid="skip"
    )

    rf = RandomForestClassifier(
        labelCol=label_col,
        featuresCol="features",
        numTrees=100,        # papier : 100 arbres
        maxDepth=10,         # papier : maxDepth=10
        seed=42
    )

    pipeline = Pipeline(stages=[imputer, assembler, rf])
    return pipeline

# ============================================================
# CELLULE 4 — Fonction d'évaluation
# Métriques papier Section 2.4 :
# Acc, Rec_o (on-time recall), Rec_d (delayed recall)
# ============================================================

def evaluate(predictions, label_col):
    """
    Calcule Acc, Rec_o, Rec_d
    selon les définitions du papier (Équations 1 et 2)
    """
    eval_acc = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="accuracy"
    )

    # TP, TN, FP, FN depuis la matrice de confusion
    tp = predictions.filter(
        (col(label_col) == 0) & (col("prediction") == 0)
    ).count()
    tn = predictions.filter(
        (col(label_col) == 1) & (col("prediction") == 1)
    ).count()
    fp = predictions.filter(
        (col(label_col) == 1) & (col("prediction") == 0)
    ).count()
    fn = predictions.filter(
        (col(label_col) == 0) & (col("prediction") == 1)
    ).count()

    acc   = (tp + tn) / (tp + tn + fp + fn) \
            if (tp + tn + fp + fn) > 0 else 0
    rec_o = tp / (tp + fn) if (tp + fn) > 0 else 0
    rec_d = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {
        "Acc":   round(acc   * 100, 1),
        "Rec_o": round(rec_o * 100, 1),
        "Rec_d": round(rec_d * 100, 1),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    }

# ============================================================
# CELLULE 5 — Entraînement + Évaluation sur tous les datasets
# Table VII du papier : D1, D2, D3, D4 × th=15 et th=60
# ============================================================

OUTPUT = "/Volumes/workspace/default/outputs/"

configs = [
    ("D1_th15", "label_15"),
    ("D2_th15", "label_15"),
    ("D3_th15", "label_15"),
    ("D4_th15", "label_15"),
    ("D1_th60", "label_60"),
    ("D2_th60", "label_60"),
    ("D3_th60", "label_60"),
    ("D4_th60", "label_60"),
]

all_results = []

for name, label_col in configs:
    print(f"\n{'='*55}")
    print(f"  ▶ {name}  —  label : {label_col}")
    print(f"{'='*55}")

    # Charger train / test
    df_train = spark.read.parquet(f"{OUTPUT}{name}_train/")
    df_test  = spark.read.parquet(f"{OUTPUT}{name}_test/")

    # Garder seulement les features existantes dans ce dataset
    existing = df_train.columns
    features = [f for f in ALL_FEATURES if f in existing]
    print(f"  Features disponibles : {len(features)}")

    if len(features) == 0:
        print("  ⚠️  Aucune feature météo — skip !")
        continue

    # Construire et entraîner le pipeline
    pipeline = build_pipeline(label_col, features)

    print(f"  🔧 Entraînement... "
          f"(train={df_train.count():,} tuples)")
    model = pipeline.fit(df_train)

    # Prédiction sur test
    print(f"  🔍 Prédiction... "
          f"(test={df_test.count():,} tuples)")
    predictions = model.transform(df_test)

    # Métriques
    metrics = evaluate(predictions, label_col)

    print(f"  ✅ Acc   = {metrics['Acc']}%")
    print(f"  ✅ Rec_o = {metrics['Rec_o']}%")
    print(f"  ✅ Rec_d = {metrics['Rec_d']}%")

    all_results.append({
        "Dataset":  name,
        "Label":    label_col,
        "Acc":      metrics["Acc"],
        "Rec_o":    metrics["Rec_o"],
        "Rec_d":    metrics["Rec_d"],
        "Train":    df_train.count(),
        "Test":     df_test.count(),
    })

    # Sauvegarder le modèle
    model.write().overwrite().save(
        f"{OUTPUT}model_{name}/"
    )
    print(f"  💾 Modèle sauvegardé !")

# ============================================================
# CELLULE 6 — Tableau comparatif avec le papier
# Table résultats Section 5
# ============================================================

print("\n" + "="*70)
print("   RÉSULTATS — Comparaison avec le papier (Section 5)")
print("="*70)
print(f"{'Dataset':<12} {'Acc':>7} {'Rec_o':>7} {'Rec_d':>7} "
      f"| {'Acc(p)':>7} {'Rec_d(p)':>9}")
print("-"*70)

# Valeurs de référence du papier (th=15 et th=60, dataset D2)
papier_ref = {
    "D2_th15": {"Acc": 74.2, "Rec_d": 71.8},
    "D2_th60": {"Acc": 85.8, "Rec_d": 86.9},
}

for r in all_results:
    ref = papier_ref.get(r["Dataset"], {})
    acc_p   = ref.get("Acc",   "—")
    recd_p  = ref.get("Rec_d", "—")
    print(f"{r['Dataset']:<12} "
          f"{r['Acc']:>6}% {r['Rec_o']:>6}% {r['Rec_d']:>6}% "
          f"| {str(acc_p):>6}% {str(recd_p):>8}%")

print("="*70)
print("(p) = valeurs papier Belcastro et al. 2016")

# ============================================================
# CELLULE 7 — Sauvegarder résultats CSV
# ============================================================

df_results = spark.createDataFrame(all_results)
df_results.coalesce(1).write.mode("overwrite").csv(
    f"{OUTPUT}ml_results/",
    header=True
)
print(f"\n✅ Résultats sauvegardés → {OUTPUT}ml_results/")
print("\n✅ Notebook 04 TERMINÉ !")